### Titanic - missing values

Łukasz Całka <br>
Uczenie Maszynowe

In [1]:
import pandas as pd

Wczytanie danych:

In [26]:
kolumny = [
    "pclass", "survived", "name", "sex", "age", "sibsp",
    "parch", "ticket", "fare", "cabin", "embarked",
    "boat", "body", "home.dest"
]

df = pd.read_csv(
    "Zbiór danych Titanic.arff",
    skiprows=17,
    header=None,
    names=kolumny,
    na_values="?"
)



In [39]:
display(df.head(20))

print(len(df))

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
5,1,1,"Anderson, Mr. Harry",male,48.0000,0,0,19952,26.5500,E12,S,3,NaN,"New York, NY"
6,1,1,"Andrews, Miss. Kornelia Theodosia",female,63.0000,1,0,13502,77.9583,D7,S,10,NaN,"Hudson, NY"
7,1,0,"Andrews, Mr. Thomas Jr",male,39.0000,0,0,112050,0.0000,A36,S,NaN,NaN,"Belfast, NI"
8,1,1,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",female,53.0000,2,0,11769,51.4792,C101,S,D,NaN,"Bayside, Queens, NY"
9,1,0,"Artagaveytia, Mr. Ramon",male,71.0000,0,0,PC 17609,49.5042,NaN,C,NaN,22.0,"Montevideo, Uruguay"


1309


## Cechy zbioru

Zbiór zawiera 14 cech, są to:
* pclass - klasa podróży (1, 2 lub 3), wyznaczik statusu ekonomicznego pasażerów
* survived - informacja czy pasażer przeżył (1) czy zginął (0)
* name - imię i nazwisko a także tytuły (Mr. Mrs. itp.)
* sex - płeć pasażera
* age - wiek w latach
* sibsp - liczba rodzeństwa i małżonków danego pasażera na pokładzie
* parch - liczba rodziców lub dzieci danego posażera na pokładzie
* ticket - numer biletu
* fare - cena biletu (w funtach)
* cabin - numer kabiny
* embarked - skrót nazwy portu, w którym pasażer wsiadł na pokład, C (Cherbourg), Q (Queenstown), S (Southampton)
* boat - numer szalupy ratunkowej, do której trafił pasażer
* body - numer identyfikacyjny wyłowionego ciała
* home.dest - miejsce zamieszkania wraz z ewentualnie miejscem docelowym podróży

<br><br>
W oczy rzuca się od razu spora liczba brakująych danych zwłaszcza w kolumnach boat oraz body.

In [23]:
braki_suma = df.isnull().sum()
print("Suma braków:\n", braki_suma)

braki_procent = df.isnull().mean() * 100
print("\nProcent braków:\n", braki_procent)

Suma braków:
 pclass          0
survived        0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

Procent braków:
 pclass        0.000000
survived      0.000000
name          0.000000
sex           0.000000
age          20.091673
sibsp         0.000000
parch         0.000000
ticket        0.000000
fare          0.076394
cabin        77.463713
embarked      0.152788
boat         62.872422
body         90.756303
home.dest    43.086325
dtype: float64


Zbiór danych należało zmodyfikować, przed wykonaniem tych operacji. Dokonano tego przy wczytaniu danych (w komórce 4) wykorzystując parametr na_values, który podstawiał NaN za znaki zapytania w wyjściowym zbiorze. <br> <br>

Najwięcej braków obserwujemy w kolumnie body, prawie 90%, z wody wyłowiono tylko ułamek ofiar. <br>
Dużo braków - 77% jest w numerze kabiny. <br>
Boat, tutaj ponad 60%, wielu pasażerów nie weszło do łodzi ratunkowych.<br>
Home.dest - 43% braków. <br>
Age - 20% braków co wydaje się dość problematyczne, wiek zdaje się być potencjalnie dobry do przewidywania przeżywalności (ratowano dzieci). <br>
Embarked oraz fare mają pojedyncze brakujące wartości, pozostałe cechy nie posiadają braków danych.

In [34]:
print(df['body'].isnull().groupby(df['survived']).mean())

survived
0    0.850433
1    1.000000
Name: body, dtype: float64


W przypadku zmiennej body obserwujemy spodziewany rozkład braków. Osoby, które przeżyły (survived=1) nie mogły otrzymać numeru ciała, stąd występuje 100% braków. Wskazuje to, iż braki są MNAR.


In [35]:
print(df['boat'].isnull().groupby(df['survived']).mean())

survived
0    0.988875
1    0.046000
Name: boat, dtype: float64


Dla cechy boat również mamy spodziewane wyniki, osoby, które nie przeżyły w większości nie były w szalupie, stąd duża liczba braków (98%), z kolei osoby, które przeżyły w większości były w szalupie stąd tam mniej niż 5% braków, znów mamy MNAR.

In [38]:
print(df['cabin'].isnull().groupby(df['pclass']).mean())
print(df['home.dest'].isnull().groupby(df['pclass']).mean())
print(df['age'].isnull().groupby(df['pclass']).mean())

pclass
1    0.207430
2    0.916968
3    0.977433
Name: cabin, dtype: float64
pclass
1    0.105263
2    0.057762
3    0.724965
Name: home.dest, dtype: float64
pclass
1    0.120743
2    0.057762
3    0.293371
Name: age, dtype: float64


Dla pozostałych 3 zmiennych braki skorelowane są z zamożnością pasażerów (klasą podróży). W przypadku numeru kabiny przywoite dane (jedynie 20% braków) są tylko dla 1 klasy, pozstałe mają ponad 90% brakujących danych. Dla cechy home.dest mamy bardzo mało braków dla 1 i 2 klasy oraz ponad 70% dla klasy 3.  W przypadku wieku znów klasa 3 wybija się, posiadając prawie 30% brakujących danych. Dane zbierane/zachowane dla najuboższych pasażerów były najmniej kompletne, braki te są, więć MAR, gdyż ich występowanie zależy od innych cech.

W przypadku braków w kolumnie fare oraz embarked możemy je potraktować jako MCAR, w zbiorze ponad 1300 obserwacji brak w postaci 1 czy dwóch wartości jest bardzo prawdopodobnie kompletnie przypadkowym błędem.

## Co zrobić z brakującymi wartościami? <br>

Jeśli chodzi o bardzo puste kolumny - body oraz boat, powinniśmy je usunąć ze zbioru. Jeśli chcemy przewidywać przeżywalność osób na statku to te dwie informacje zakłamią wyniki. Jeśli będzie istniał numer szalupy dla danego pasażera to niemal na pewno przeżył, informacja o numerze ciała daje z kolei 100% pewności, że pasażer zginął. <br>
Kolumna cabin posiada bardzo dużo braków i jej uzupełnianie nie ma sensu ani sensownej metody. Moglibyśmy albo ją usunac (bardzo podobną informację niesie już kolumna pclass) albo przekształcić na typ binarny - czy pasażer posiadał przypisaną kabinę (0 lub 1). <br>
Dla zmiennej home.dest ciężko znaleźć sensowny sposób uzupełniania, można by wpisać "Nieznane"/"Unknown" co niosłoby za sobą informację o prawdopodobnie niższym statusie społecznym takiego pasażera (korelacje braków z klasą podróży). <br>
Najważniejsze działania są dla zmiennej age, gdyż ona może mieć kluczowe znaczenie przy przewidywaniu przeżywalności (ratowano dzieci). Trzeba tu wypełnić braki. W najprostszym sposobie można to zrobić średnią lub medianą globalną (po weryfikacji skośności rozkładu). Lepszą opcją byłoby jednak podzielić dane na klasy na przykład względem kolumn pclass oraz sex i dopiero w tych podzbiorach uzupełniać średnią lub medianą "lokalną". <br>
W przypadku embarked najlepiej uzupełnić opcją najczęściej występującą, więc najbardziej prawdopodobną. Dla fare z kolei znów można użyć średniej lub mediany globalnej lub dopasowanej do danej klasy podróży pclass.

W zadaniu wykorzystano narzędzia generatywnej sztucznej inteligencji w celu szybkiego wyekstraktowania nazw kolumn z pliku ze zbiorem danych a także do poprawnego zredagowania wniosków obserwowanych w zbiorze danych. 